# Deploying to a Cloud Platform

---

In this notebook, we will deploy our Dockerized Iris Prediction API to **Railway**, a PaaS that deploys directly from a GitHub repository.

We will cover:

- Preparing the repository for deployment.
- Connecting Railway to GitHub.
- Configuring the deployment.
- Testing the live API.
- Understanding what happens behind the scenes.

> ⚠️ **Note:** We use Railway as the example because it's the fastest path from GitHub repo to live URL. The concepts (environment variables, health checks, port configuration) transfer to any platform.

---

## 1. Prerequisites

Before deploying, make sure you have:

- [x] A **GitHub repository** with your code pushed (you already have `kocyigit-dsml`)
- [x] The **Dockerfile** and **requirements.txt** in `04_mlops/03_containerization/`
- [x] The **model artifact** (`iris_pipeline.joblib`) and **app code** copied into the `03_containerization/` directory
- [ ] A **Railway account** — sign up at [railway.app](https://railway.app/) using your GitHub account

---

## 2. Repository Structure for Deployment

Railway (and most PaaS platforms) deploy from a directory in your repository. It looks for a `Dockerfile` and builds the image automatically.

Our deploy-ready directory is `04_mlops/03_containerization/`:

```
03_containerization/
├── Dockerfile
├── docker-compose.yml
├── requirements.txt
├── app/
│   ├── main.py
│   ├── schemas.py
│   └── model_loader.py
└── models/
    └── iris_pipeline.joblib
```

This directory is **self-contained**: it has the Dockerfile, the code, the dependencies, and the model. The platform doesn't need anything from the rest of the repository.

---

## 3. Step-By-Step: Deploy to Railway


### Step 1: Create a New Project

1. Go to [railway.app/dashboard](https://railway.app/dashboard)
2. Click **"New Project"**
3. Select **"Deploy from GitHub Repo"**
4. Authorize Railway to access your GitHub repositories
5. Select the `kocyigit-dsml` repository

### Step 2: Configure the Root Directory

Railway needs to know which directory contains the Dockerfile:

1. Go to your service's **Settings** tab
2. Under **"Root Directory"**, set it to: `04_mlops/03_containerization`
3. Railway will now look for the `Dockerfile` in that directory

### Step 3: Set Environment Variables

1. Go to the **"Variables"** tab
2. Add the following variable:

| Key | Value |
| :--- | :--- |
| `MODEL_PATH` | `/app/models/iris_pipeline.joblib` |
| `PORT` | `8000` |

> Some platforms require a `PORT` variable. Railway auto-detects it from the `EXPOSE` instruction in your Dockerfile, but it's good practice to set it explicitly.

### Step 4: Deploy

1. Click **"Deploy"** (or Railway auto-deploys when you push to `main`)
2. Watch the build logs — you'll see Docker building the image layer by layer
3. Once the build completes, Railway starts the container
4. You'll see `✅ Model loaded successfully.` in the deploy logs

### Step 5: Get Your Public URL

1. Go to the **"Settings"** tab
2. Under **"Networking"**, click **"Generate Domain"**
3. Railway gives you a URL like: `https://iris-api-production-xxxx.up.railway.app`

Your API is now live on the internet! 🎉

---

## 4. Testing the Live API

Replace `YOUR_URL` with the Railway-generated URL:

### Health Check

In [ ]:
curl https://YOUR_URL.up.railway.app/health

Expected:
```json
{"status": "healthy", "model_loaded": true}
```

### Prediction

In [ ]:
curl -X POST https://YOUR_URL.up.railway.app/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'

Expected:
```json
{"prediction": "setosa", "prediction_id": 0, "probabilities": {"setosa": 0.97, ...}}
```

### Swagger UI

Visit `https://YOUR_URL.up.railway.app/docs` in your browser to get the full interactive documentation.

### Python Client

In [4]:
import requests

YOUR_URL = "kocyigit-dsml-production"

response = requests.post(
    f"https://{YOUR_URL}.up.railway.app/predict",
    json={"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.2, "petal_width": 2.3}
)
print(response.json())

{'prediction': 'virginica', 'prediction_id': 2, 'probabilities': {'setosa': 0.0, 'versicolor': 0.04, 'virginica': 0.96}}



---

## 5. What Happens Behind the Scenes

1. Railway clones your GitHub repository
2. It navigates to the root directory (04_mlops/03_containerization)
3. It finds the Dockerfile and runs `docker build`
4. The image is stored in Railway's internal registry
5. Railway starts a container from the image
6. It injects environment variables (MODEL_PATH, PORT)
7. It routes external HTTPS traffic to the container's port 8000
8. SSL/TLS certificate is provisioned automatically

You didn't have to:
- Set up a server
- Configure Nginx
- Generate SSL certificates
- Set up DNS
- Manage a container registry

That's the value of PaaS.

---

## 6. Adapting to Other Platforms

The deployment pattern is nearly identical across platforms. Here's what changes:

### Google Cloud Run

In [ ]:
# Build and push the image to Google Container Registry
gcloud builds submit --tag gcr.io/YOUR_PROJECT/iris-api

# Deploy to Cloud Run
gcloud run deploy iris-api \
  --image gcr.io/YOUR_PROJECT/iris-api \
  --platform managed \
  --port 8000 \
  --set-env-vars MODEL_PATH=/app/models/iris_pipeline.joblib \
  --allow-unauthenticated

### Render

1. Connect GitHub repo
2. Set root directory to `04_mlops/03_containerization`
3. Render auto-detects the Dockerfile
4. Set environment variables in the dashboard

### Fly.io

In [ ]:
# From the 03_containerization/ directory
fly launch
fly deploy

The core workflow is always the same: **give the platform a Dockerfile, set environment variables, deploy**.

---

## 7. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **Root Directory** | Tell the platform which folder contains your Dockerfile. |
| **Environment Variables** | Set `MODEL_PATH` and `PORT` in the platform's dashboard, not in code. |
| **Public URL** | The platform generates an HTTPS URL with automatic SSL. |
| **Auto-deploy** | Most platforms re-deploy automatically when you push to `main`. |
| **Portability** | The same Dockerfile works on Railway, Cloud Run, Render, Fly.io, etc. |
| **PaaS value** | No servers, no Nginx, no SSL, no DNS. Just push and deploy. |

---


**Next:** [CI/CD Basics with GitHub Actions](./03_ci_cd_basics_with_github_actions.ipynb) — Automating tests and deployments on every push.